# FITS Blazar Data Extraction

This notebook reads the `gll_psc_v35.fit` FITS file, filters for sources classified as BLL, FSRQ, or BCU, and saves their complete catalog data to a CSV file.

In [ ]:
import astropy.io.fits as fits
import pandas as pd

## Load FITS File

Open the FITS file and access the catalog data and headers.

In [ ]:
# Open the FITS file
file_name = './raw_data/gll_psc_v35.fit' 
hdul = fits.open(file_name)

# Get the data for the sources and the headers
data_headers = hdul[1].columns.names
catalog_data = hdul[1].data

# Display information about the FITS file structure
hdul.info()
print(f"\nSuccessfully loaded {len(catalog_data)} sources.")

Filename: ./Data/gll_psc_v35.fit
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      24   ()      
  1  LAT_Point_Source_Catalog    1 BinTableHDU    456   7195R x 79C   ['18A', 'I', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'I', '18A', 'E', 'E', 'E', 'E', 'E', 'E', '17A', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', '8E', '16E', '8E', '8E', 'E', 'E', 'E', 'E', 'E', 'E', 'D', 'E', '14E', '28E', '14E', '18A', '18A', '18A', '18A', '18A', '18A', 'A', '30A', '5A', '10A', '28A', '30A', 'E', 'E', 'D', 'D', 'E', 'I']   
  2  ExtendedSources    1 BinTableHDU     74   82R x 11C   [17A, E, E, E, E, 11A, E, E, E, 11A, 24A]   
  3  ROIs          1 BinTableHDU    121   1991R x 10C   [I, E, E, E, E, E, E, E, E, E]   
  4  Components    1 BinTableHDU     53   19R x 9C   [E, E, I, I, E, E, E, I, I]   
  5  EnergyBounds    1 BinTableHDU     73   20R x 10C   [E, E, E, I, 

# Filter Sources by Class

Iterate through all sources and select only those classified as `bll`, `fsrq`, or `bcu` in either the `CLASS1` or `CLASS2` fields.

In [4]:
# Define the classifications we are interested in
interested_classes = ['bll', 'fsrq', 'bcu']
selected_sources = []

# Loop through each record in the catalog data
for source_record in catalog_data:
    # Check if either classification field matches our list of interest.
    # .strip() removes leading/trailing whitespace and .lower() makes the check case-insensitive.
    if (source_record['CLASS1'].strip().lower() in interested_classes) or \
       (source_record['CLASS2'].strip().lower() in interested_classes):
        selected_sources.append(source_record)

print(f"Found {len(selected_sources)} sources matching the criteria: {interested_classes}")

Found 3933 sources matching the criteria: ['bll', 'fsrq', 'bcu']


# Save Filtered Data to CSV

Convert the list of filtered sources into a pandas DataFrame and save it to a CSV file.

In [ ]:
# Proceed only if we found sources to save
if selected_sources:
    data_for_df = []
    for record in selected_sources:
        data_for_df.append(list(record))
    
    # Create the DataFrame from the clean list of lists, and pass the headers
    output_df = pd.DataFrame(data_for_df, columns=data_headers)
    
    # Define the output filename
    output_filename = 'blazar_catalog_v35.csv'
    
    # Save the DataFrame to a CSV file
    output_df.to_csv(output_filename, index=False)
    
    print(f"Successfully saved the data with headers to '{output_filename}'")
    
    # Display the first 5 rows of the new DataFrame as a preview
    print("\n--- First 5 rows of the saved data ---")
    display(output_df.head())
else:
    print("No sources were found to save.")

c:\Users\shres\AppData\Local\Programs\Python\Python310\lib\site-packages\pandas\core\internals\blocks.py:2538: RuntimeWarning: invalid value encountered in cast
  values = values.astype(str)


Successfully saved the data with headers to 'blazar_catalog.csv'

--- First 5 rows of the saved data ---


,Source_Name,DataRelease,RAJ2000,DEJ2000,GLON,GLAT,Conf_68_SemiMajor,Conf_68_SemiMinor,Conf_68_PosAng,Conf_95_SemiMajor,...,CLASS1,CLASS2,ASSOC1,ASSOC2,ASSOC_PROB_BAY,ASSOC_PROB_LR,RA_Counterpart,DEC_Counterpart,Unc_Counterpart,Flags
0,4FGL J0001.2+4741,1,0.3126,47.685902,114.250198,-14.338059,0.036880,0.033180,-45.900002,0.0598,...,bcu,,B3 2358+474,,0.996097,0.938563,0.329341,47.700201,8.400000e-07,0
1,4FGL J0001.2-0747,1,0.3151,-7.797100,89.032722,-67.305008,0.018440,0.017577,64.099998,0.0299,...,bll,,PMN J0001-0746,,0.997014,0.932932,0.325104,-7.774145,1.800000e-07,0
2,4FGL J0001.4-0010,3,0.3717,-0.169900,96.891991,-60.491329,0.036510,0.036325,-52.740002,0.0592,...,bll,,FBQS J0001-0011,,0.991603,0.817716,0.339540,-0.194420,4.500000e-03,0
3,4FGL J0001.5+2113,1,0.3815,21.218300,107.649353,-40.167652,0.026026,0.023991,-60.520000,0.0422,...,fsrq,,TXS 2358+209,,0.998049,0.958182,0.384877,21.226743,3.400000e-07,0
4,4FGL J0001.6-4156,1,0.4165,-41.942501,334.226349,-72.028549,0.042678,0.032378,44.090000,0.0692,...,bcu,,2MASS J00013275-4155252,,0.996336,0.854569,0.386480,-41.923705,6.900000e-05,0


# Close the FITS File

Release the file from memory.

In [11]:
# Close the FITS file to free up system resources
hdul.close()
print("FITS file closed.")

FITS file closed.
